In [18]:
from peft import __version__

print(__version__)

0.19.1


In [19]:
!pip install -U trl

  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached datasets-5.0.0-py3-none-any.whl (555 kB)
Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (48.9 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [1]:
import time
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [2]:

from peft import PeftModel

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-sft"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [4]:
from peft import (
    PrefixTuningConfig,
    TaskType,
    get_peft_model
)

In [5]:
prefix_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,

    num_virtual_tokens=20,

    inference_mode=False
)

In [6]:
model = get_peft_model(
    model,
    prefix_config
)

In [7]:
model.print_trainable_parameters()

trainable params: 122,880 || all params: 494,155,648 || trainable%: 0.0249


In [8]:
from datasets import Dataset
import pandas as pd

train_df = pd.read_excel("/content/cleaned_dataset.xlsx")
val_df = pd.read_csv("validation.csv")

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [9]:
def format_example(example):

    messages = [
        {
            "role": "system",
            # "content": "You are answering questions about Vishnu."
           "content": (
            "You are Vishnu's personal AI assistant. "
            "Answer questions about Vishnu."
        )
        },
        {
            "role": "user",
            "content": example["Clean Question"]
        },
        {
            "role": "assistant",
            "content": example["Clean Answer"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [10]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [11]:
train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [12]:
train_dataset = train_dataset.map(tokenize_function)
val_dataset = val_dataset.map(tokenize_function)

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [13]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./prefix_tuning",

    learning_rate=1e-2,

    num_train_epochs=20,

    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,

    logging_steps=1,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none"
)

In [14]:
from trl import SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.678559,1.015347,1.024820,29440.000000,0.866274
2,0.572239,0.665513,0.751106,58880.000000,0.890980
3,0.231659,0.537401,0.627278,88320.000000,0.902353
4,0.573829,0.462826,0.558427,117760.000000,0.912157
5,0.651479,0.407247,0.482685,147200.000000,0.919216
6,0.157466,0.364801,0.438652,176640.000000,0.923922
7,0.304262,0.331968,0.405116,206080.000000,0.929804
8,0.213353,0.298485,0.395130,235520.000000,0.935686
9,0.145497,0.275701,0.361191,264960.000000,0.936078
10,0.424066,0.258352,0.343831,294400.000000,0.941176


TrainOutput(global_step=1160, training_loss=0.4113177875185321, metrics={'train_runtime': 661.0601, 'train_samples_per_second': 3.479, 'train_steps_per_second': 1.755, 'total_flos': 1264382450073600.0, 'train_loss': 0.4113177875185321, 'epoch': 20.0})

In [23]:
trainer.save_model("./full_ft_qwen_prefix_tuning")
tokenizer.save_pretrained("./full_ft_qwen_prefix_tuning")

('./full_ft_qwen_prefix_tuning/tokenizer_config.json',
 './full_ft_qwen_prefix_tuning/chat_template.jinja',
 './full_ft_qwen_prefix_tuning/tokenizer.json')

In [27]:
messages = [
    {
        "role": "system",
        "content": (
            "You are Vishnu's personal AI assistant. "
            "Answer questions about Vishnu."
        )
    },
    {
        "role": "user",
        "content": "why do you like movies so much vishnu?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
You are Vishnu's personal AI assistant. Answer questions about Vishnu.
user
why do you like movies so much vishnu?
assistant
I watched every movie, but I just saw many.
